## Kelompok 2

Anggota Kelompok (Alfabetis):
1. Azis Al Risal
2. Bagas Dwi Saputra
3. Desta Ega Fatima
4. Jonathan Steve Roland
5. Nayla Khairusyi Shabrina

---

> _Dev Note_
>
> _Sebelum di running kita install package / library yang dibutuhin dulu ges, list nya ada didalam file requirements.txt, untuk install nya bisa menggunakan pip pakai command `pip install -r requirements.txt`:_

##### Persiapan Awal dan Impor Data
Bagian kode yang ini adalah bagian yang paling pertama, persiapan. Kita panggil semua library Python yang dibutuhin:

- `pandas` untuk memanipulasi data berwujud tabel.
- `matplotlib` & `seaborn` untuk membuat visualisasi dan grafik.
- `scikit-learn` untuk menyediakan algoritma Machine Learning (Random Forest & SVM), teknik scaling, dan metrik evaluasi.
- `imbalanced-learn` (SMOTE) untuk mengatasi ketidakseimbangan jumlah data pada kelas wine.

Data CSV kemudian dibaca dan ditampilkan strukturnya menggunakan fungsi `data.info()` untuk memastikan tidak ada nilai kosong (missing values) yang terlewat.

In [ ]:
import time
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Mematikan pesan warning yang mengganggu di terminal
warnings.filterwarnings("ignore")
data = pd.read_csv("wine-data.csv")

print("[INFO] 5 BARIS PERTAMA PADA DATASET")
display(data.head())
print("[INFO] INFORMASI FILE DATASET")
data.info()

##### EDA
Tahap EDA adalah mengenali data sebelum diserahkan kepada algoritma AI. Kita membuat 3 grafik utama untuk menjawab karakteristik dataset:

1. Distribusi Kategori
   - Dari grafik ini, dataset kita ada masalah, yaitu mayoritas data adalah wine kelas Average, sementara Low dan High sangat sedikit. Karena itulah kita akan pakai teknik SMOTE untuk menyeimbangkan data.
2. Heatmap Korelasi
   - Digunakan untuk memetakan hubungan tarik-menarik antar senyawa kimiawi. Misalnya, kita bisa melihat kadar alcohol memiliki warna korelasi positif yang pekat terhadap kualitas wine.
3. Boxplot
   - Memperlihatkan secara detail dan spesifik rentang persebaran kadar alkohol pada setiap kelas wine.

In [ ]:
print(f"[INFO] MENGANALISIS DATA DAN MEMBUAT DIAGRAM...")

# 1. DIAGRAM DISTRIBUSI KATEGORI
plt.figure(figsize=(8,5))
plt.title("Distribusi Kategori Wine (WineCategory)")
plt.xlabel("Kategori Wine")
plt.ylabel("Jumlah Dataset")
plt.tight_layout()
sns.countplot(data=data, x="WineCategory", hue="WineCategory", order=["Low", "Average","High"], palette="viridis", legend=False)
plt.show()

# 2. DIAGRAM HEATMAP KORELASI
corr_matrix = data.drop(columns=["WineCategory"]).corr()
plt.figure(figsize=(12,8))
plt.title("Heatmap Korelasi Fitur Kualitas Wine")
plt.tight_layout()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.show()

# 3. DIAGRAM BOXPLOT ALKOHOL VS KATEGORI WINE
plt.figure(figsize=(8,5))
plt.title("Hubungan Kadar Alkohol vs Kategori Wine")
plt.xlabel("Kategori Wine")
plt.ylabel("Kadar Alkohol")
plt.tight_layout()
sns.boxplot(data=data, x="WineCategory", y="alcohol", hue="WineCategory", order=["Low", "Average", "High"], palette="Set2", legend=False)
plt.show()
print("[INFO] GRAFIK DIBUAT...")

##### Persiapan & Preprocessing Data
Disini, data mentah diolah dan diubah agar algoritma AI bisa memprosesnya. Langkah-langkahnya adalah:

1. Pemisahan Fitur & Target: 
   - Kolom `WineCategory` dipisahkan sebagai Target (Y), dan kolom lainnya menjadi Fitur (X). Namun, kolom `quality` (angka) harus dibuang agar model AI tidak menyontek atau bias.
2. Label Encoding: 
   - Mengubah teks "Low", "Average", "High" menjadi angka (0, 1, 2) yang dimengerti AI.
3. Split Data (80:20): 
   - Membagi data menjadi porsi Training (untuk AI belajar) dan Testing (untuk menguji AI nanti). Parameter `stratify=y` memastikan porsi pembagian kelas wine tetap merata di kedua bagian.
4. Feature Scaling (StandardScaler): 
   - Pada data, terdapat kolom yang satuannya bernilai puluhan dan ada juga yang memiliki nilai desimal. Scaler menyetarakan rentang angka ini agar kolom yang angkanya besar tidak menindas kolom yang angkanya kecil saat diproses oleh algoritma.
5. SMOTE: 
   - Menggandakan data kelas minoritas (Low dan High) secara sintetis agar setara dengan kelas Average. Tanpa SMOTE, model AI akan cenderung selalu menebak Average.

In [ ]:
# Pisahkan Fitur dan Target
X = data.drop(columns=["quality", "WineCategory"]) 
le = LabelEncoder()
y = le.fit_transform(data["WineCategory"]) 

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE (Menangani data yang tidak seimbang)
X_train_balanced, Y_train_balanced = SMOTE(random_state=42).fit_resample(X_train_scaled, Y_train)

print(f"[INFO] JUMLAH BARIS DATA TRAINING SEBELUM SMOTE : {len(Y_train)}")
print(f"[INFO] JUMLAH BARIS DATA TRAINING SESUDAH SMOTE : {len(Y_train_balanced)}")

##### Model Training & Hyperparameter Tuning
Ini tahap saat AI dilatih. Kita membandingkan dua algoritma kuat: Random Forest (berbasis pepohonan keputusan) dan SVM (berbasis batas ruang margin).
Kita tidak sekadar memasukkan data begitu saja tapi kita menggunakan GridSearchCV untuk melakukan Hyperparameter Tuning. Artinya, kita memerintahkan Python untuk melakukan simulasi ratusan kali demi mencari parameter yang memberikan akurasi tertinggi secara otomatis, menggunakan validasi silang (cv=5).

In [ ]:
# TRAINING & HYPERPARAMETER TUNING
print("[INFO] MEMULAI PENCARIAN HYPERPARAMETER TERBAIK (TUNING)...")
start_time = time.time()

# ALGORITMA 1: RANDOM FOREST
randomForestParamGrid = {
    "n_estimators": [50,100,200],
    "max_depth": [None, 10,20],
}
randomForestGrid = GridSearchCV(RandomForestClassifier(random_state=42), randomForestParamGrid, cv=5, scoring="accuracy", n_jobs=-1)
randomForestGrid.fit(X_train_balanced, Y_train_balanced)
bestRandomForestModel = randomForestGrid.best_estimator_

# ALGORITMA 2: SUPPORT VECTOR MACHINE (SVM)
supportVectorMachineParamGrid = {
    "C":[0.1,1,10],
    "kernel":["linear", "rbf"]
}
supportVectorMachineGrid = GridSearchCV(SVC(random_state=42), supportVectorMachineParamGrid, cv=5, scoring="accuracy", n_jobs=-1)
supportVectorMachineGrid.fit(X_train_balanced, Y_train_balanced)
bestSupportVectorMachineGridModel = supportVectorMachineGrid.best_estimator_

print(f"PROSES SELESAI DALAM WAKTU {time.time() - start_time:.2f} detik!")
print(f"[RANDOM FOREST] HYPERPARAMETER TERBAIK  : {randomForestGrid.best_params_}")
print(f"[SVM] HYPERPARAMETER TERBAIK            : {supportVectorMachineGrid.best_params_}")

In [ ]:
# Tahap 5. Evaluasi & Kesimpulan
randomForestPrediction = bestRandomForestModel.predict(X_test_scaled)
supportVectorMachinePrediction = bestSupportVectorMachineGridModel.predict(X_test_scaled)

# Evaluasi Algoritma Random Forest
print("[INFO] EVALUASI RANDOM FOREST")
print("-"*40)
print(f"Accuracy Score      : {accuracy_score(Y_test, randomForestPrediction):.4f}")
print(f"F1-Score (Weighted) : {f1_score(Y_test, randomForestPrediction, average='weighted'):.4f}")
print("Confusion Matrix:\n", confusion_matrix(Y_test, randomForestPrediction))
print("\nClassification Report:\n", classification_report(Y_test, randomForestPrediction, target_names=le.classes_))

# Evaluasi Algoritma SVM
print("\n[INFO] EVALUASI ALGORITMA SUPPORT VECTOR MACHINE (SVM)")
print("-"*40)
print(f"Accuracy Score      : {accuracy_score(Y_test, supportVectorMachinePrediction):.4f}")
print(f"F1-Score (Weighted) : {f1_score(Y_test, supportVectorMachinePrediction, average='weighted'):.4f}")
print("Confusion Matrix:\n", confusion_matrix(Y_test, supportVectorMachinePrediction))
print("\nClassification Report:\n", classification_report(Y_test, supportVectorMachinePrediction, target_names=le.classes_))

##### Kesimpulan Akhir
Dari hasil pengujian di atas, algoritma Random Forest terbukti jauh lebih unggul dibandingkan Support Vector Machine (SVM) dalam mengklasifikasikan kualitas wine.

* `Random Forest` mencetak akurasi sebesar 85.62% dengan `F1-Score` 86.27%.
* `SVM` mencetak akurasi 75.62% dengan `F1-Score` 78.27%.